*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*

> This notebook contains the raw code for Chapter 5: Production-Ready Data Pipelines. To understand how datasets, workers, batching, and pinned transfers fit together in real training, get the full step-by-step guide on Amazon: **[Get the Book Here](https://www.amazon.fr/Mastering-PyTorch-Lightning-Step-Step-ebook/dp/B0HGNZS55V)**

A strong training loop depends on data throughput as much as model quality. The Dataset defines how to fetch one sample, while the DataLoader decides how those samples are batched, prefetched, and transferred to the device.

## 1. Crafting Custom Datasets
### Step 1: Implement and Test the Dataset Protocol

In [ ]:
import torch
from torch.utils.data import Dataset

# A Dataset defines how to fetch one sample by index.
# __len__() returns the total number of samples.
# __getitem__() returns one (input, target) pair for the given index.
class TensorPairDataset(Dataset):
    def __init__(self, features, targets):
        if len(features) != len(targets):
            raise ValueError("Features and targets must align")
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.targets[index]

features = torch.randn(10, 4)
targets = torch.randint(0, 3, (10,))
train_dataset = TensorPairDataset(features, targets)

sample_features, sample_target = train_dataset[0]
assert len(train_dataset) == 10
assert sample_features.shape == (4,)
assert sample_target.ndim == 0

### Step 2: Extend the Protocol to Lazy Image Loading

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

# Lazy loading keeps only metadata in memory and reads files only when __getitem__() is called.
# This trades storage I/O for lower RAM usage and is essential for large datasets.
class VisionDataset(Dataset):
    def __init__(self, image_dir, labels_dict, transform=None):
        self.image_dir = image_dir

        # Keep only lightweight metadata in RAM.
        self.image_filenames = sorted(labels_dict)
        self.labels = labels_dict
        self.transform = transform

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        # Read the image only when this sample is requested.
        img_name = self.image_filenames[idx]
        img_path = os.path.join(self.image_dir, img_name)

        # Normalize the channel layout and close the file promptly.
        with Image.open(img_path) as source:
            image = source.convert("RGB")

        label = self.labels[img_name]

        if self.transform is not None:
            image = self.transform(image)

        return image, label

### Step 3: Choose an Appropriate Structured-Data Backend
## 2. Orchestrating the Pipeline: The `DataLoader`
### Step 1: Verify Batching in the Main Process

In [ ]:
from torch.utils.data import DataLoader

# DataLoader handles batching, collation, shuffling, and iteration.
# Always start with num_workers=0 to test on the main process; multiprocessing complicates debugging.
debug_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

batch_features, batch_targets = next(iter(debug_loader))

assert batch_features.shape == (4, 4)
assert batch_targets.shape == (4,)
assert len(debug_loader) == 3

### Step 2: Add Worker Processes and Prefetching

In [ ]:
import os
from torch.utils.data import DataLoader

# num_workers > 0 launches worker processes for parallel data loading.
# persistent_workers=True keeps workers alive between iterations; prefetch_factor controls queue depth.
# Only enable these after the single-process loader is correct.
num_workers = min(4, os.cpu_count() or 1)

worker_options = dict(
    dataset=train_dataset,
    batch_size=4,
    shuffle=False,
    drop_last=True,
    num_workers=num_workers,
)

if num_workers > 0:
    worker_options.update(
        persistent_workers=True,
        prefetch_factor=2,
    )

parallel_loader = DataLoader(**worker_options)

assert parallel_loader.num_workers == num_workers
assert len(parallel_loader) == 2

if num_workers > 0:
    assert parallel_loader.persistent_workers
    assert parallel_loader.prefetch_factor == 2

### Step 3: Add Pinned Memory and Device Transfer

In [ ]:
import torch
from torch.utils.data import DataLoader

# pin_memory=True locks batch tensors in page-locked host memory.
# This enables efficient asynchronous GPU transfers when combined with non_blocking=True.
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu", index=0)

transfer_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    drop_last=True,
    num_workers=0,
    pin_memory=use_cuda,
)

batch_features, batch_targets = next(iter(transfer_loader))
batch_features = batch_features.to(device, non_blocking=use_cuda)
batch_targets = batch_targets.to(device, non_blocking=use_cuda)

assert batch_features.device == device
assert batch_targets.device == device
assert len(transfer_loader) == 2